# MeetMind — Fine-tuning de CamemBERT (décision / action / information)

**Yousra Erraki · 2026-2027**

Notebook **autonome** : les données d'entraînement (431 phrases) et de test (70 phrases) sont incluses.
Rien à charger.

### Mode d'emploi
1. Menu **Exécution → Modifier le type d'exécution → GPU (T4)** → Enregistrer
2. Menu **Exécution → Tout exécuter** (Ctrl + F9)
3. Attendre ~5 minutes. Le fichier **`camembert-meetmind.zip`** se télécharge à la fin.
4. Sur le PC : dézipper dans `Desktop\meetmind\models\` puis relancer l'app.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU" 

In [ ]:
!pip install -q datasets accelerate 2>/dev/null
import transformers, torch; print("transformers", transformers.__version__, "| GPU :", torch.cuda.is_available())

## Données (corpus synthétique MeetMind)
431 phrases d'entraînement · 70 phrases de test issues de 5 réunions jamais vues.

In [ ]:
DATA = {"train": [{"text": "Bonjour à tous, l'objectif de ce matin est de préparer le lancement de la version 2.", "label": "information"}, {"text": "Les tests de charge sont terminés, le serveur tient deux mille utilisateurs simultanés.", "label": "information"}, {"text": "Très bien, et côté design, où en est-on ?", "label": "information"}, {"text": "Il reste deux écrans à finaliser, l'onboarding et la page de paiement.", "label": "information"}, {"text": "Bon, on retient la date du 1er décembre pour le lancement.", "label": "decision"}, {"text": "Ça me paraît jouable si le design est prêt à temps.", "label": "information"}, {"text": "Lina, tu termines les deux écrans d'ici jeudi ?", "label": "action"}, {"text": "Oui, je m'en occupe.", "label": "action"}, {"text": "Je vais préparer le plan de retour arrière en cas de problème.", "label": "action"}, {"text": "Pour la communication, on avait parlé d'une vidéo de démonstration.", "label": "information"}, {"text": "Une vidéo coûte cher, environ cinq mille dirhams avec un prestataire.", "label": "information"}, {"text": "Finalement on fait la vidéo en interne, pas de prestataire.", "label": "decision"}, {"text": "Lina se charge du script de la vidéo pour lundi prochain.", "label": "action"}, {"text": "On pourrait aussi publier un article de blog le jour du lancement.", "label": "information"}, {"text": "Bonne idée, on en parlera à la prochaine réunion.", "label": "information"}, {"text": "Adam, pense à mettre à jour la documentation de l'API avant le lancement.", "label": "action"}, {"text": "Merci à tous, bonne journée.", "label": "information"}, {"text": "Bonjour, nous allons parler des examens blancs et de la sortie scolaire.", "label": "information"}, {"text": "Les examens blancs de mathématiques sont prévus pour la semaine du 7 décembre.", "label": "information"}, {"text": "En physique, nous n'avons pas encore fixé de date.", "label": "information"}, {"text": "Il est décidé que tous les examens blancs auront lieu la même semaine.", "label": "decision"}, {"text": "Monsieur Chraibi, vous pouvez envoyer votre sujet avant le 30 novembre ?", "label": "action"}, {"text": "Entendu, je vous l'envoie avant la fin du mois.", "label": "action"}, {"text": "Pour la sortie au musée, nous avons trente-deux élèves inscrits.", "label": "information"}, {"text": "Le bus coûte mille deux cents dirhams pour la journée.", "label": "information"}, {"text": "Nous validons la sortie au musée pour le vendredi 11 décembre.", "label": "decision"}, {"text": "Je me charge de collecter les autorisations parentales.", "label": "action"}, {"text": "Il faut réserver le bus cette semaine.", "label": "action"}, {"text": "Je peux m'occuper de la réservation du bus.", "label": "action"}, {"text": "Est-ce que les parents accompagnateurs sont déjà prévus ?", "label": "information"}, {"text": "Pas encore, on verra selon les réponses.", "label": "information"}, {"text": "Merci à tous pour votre présence.", "label": "information"}, {"text": "Bonjour, on fait le point sur les gardes du mois de décembre.", "label": "information"}, {"text": "Nous manquons de deux infirmières pour les nuits du week-end.", "label": "information"}, {"text": "C'est un problème qui revient chaque année pendant les fêtes.", "label": "information"}, {"text": "Nous allons faire appel à l'agence d'intérim pour les nuits de décembre.", "label": "decision"}, {"text": "Madame Rachidi, contactez l'agence avant mercredi s'il vous plaît.", "label": "action"}, {"text": "D'accord, je les appelle demain matin.", "label": "action"}, {"text": "Le nouveau moniteur cardiaque est arrivé mais personne n'est formé.", "label": "information"}, {"text": "Je vais organiser une formation pour l'équipe la semaine prochaine.", "label": "action"}, {"text": "Combien de temps dure la formation ?", "label": "information"}, {"text": "Environ deux heures par groupe.", "label": "information"}, {"text": "Ok, la formation sera obligatoire pour toute l'équipe de jour.", "label": "decision"}, {"text": "Docteur Tazi, merci d'envoyer le planning des groupes à tout le monde.", "label": "action"}, {"text": "Les patients se plaignent aussi du temps d'attente aux urgences.", "label": "information"}, {"text": "On abordera ce point à la réunion de direction.", "label": "information"}, {"text": "Bon courage à tous.", "label": "information"}, {"text": "Salut tout le monde, on doit organiser le tournoi de football du mois prochain.", "label": "information"}, {"text": "On a déjà douze équipes inscrites, il en manque quatre.", "label": "information"}, {"text": "Le terrain municipal est libre le 12 et le 19 décembre.", "label": "information"}, {"text": "On choisit le samedi 19 décembre pour le tournoi.", "label": "decision"}, {"text": "Il faut encore trouver un sponsor pour les maillots.", "label": "information"}, {"text": "Hiba, tu peux contacter la boulangerie du quartier pour le sponsoring ?", "label": "action"}, {"text": "Pas de problème, je passe les voir demain.", "label": "action"}, {"text": "Je m'occupe de la réservation du terrain auprès de la mairie.", "label": "action"}, {"text": "L'année dernière, on avait eu des problèmes d'arbitrage.", "label": "information"}, {"text": "Cette année, on prend des arbitres officiels de la ligue.", "label": "decision"}, {"text": "Ça coûte trois cents dirhams par match.", "label": "information"}, {"text": "Omar, tu me fais un budget complet pour vendredi.", "label": "action"}, {"text": "Et les inscriptions des quatre équipes manquantes ?", "label": "information"}, {"text": "Je publie une annonce sur la page Facebook ce soir.", "label": "action"}, {"text": "Merci les amis, à la semaine prochaine.", "label": "information"}, {"text": "On a reçu les retours du client sur la maquette du site e-commerce.", "label": "information"}, {"text": "Ils trouvent la page d'accueil trop chargée.", "label": "information"}, {"text": "Ils veulent aussi un mode sombre, mais ce n'était pas prévu dans le devis.", "label": "information"}, {"text": "Le mode sombre sera facturé en supplément, c'est acté.", "label": "decision"}, {"text": "Yassir, tu simplifies la page d'accueil pour jeudi.", "label": "action"}, {"text": "Ok, je fais une nouvelle version d'ici jeudi.", "label": "action"}, {"text": "Le client demande aussi quand le site sera en ligne.", "label": "information"}, {"text": "Avec les corrections, on peut viser la mi-décembre.", "label": "information"}, {"text": "On annonce au client une mise en ligne le 15 décembre.", "label": "decision"}, {"text": "Je rédige le devis pour le mode sombre aujourd'hui.", "label": "action"}, {"text": "Merci d'envoyer le devis au client avant demain midi.", "label": "action"}, {"text": "Est-ce qu'on garde la même police d'écriture ?", "label": "information"}, {"text": "Je pense que oui, mais demande-leur leur avis.", "label": "information"}, {"text": "Je vais appeler le client pour confirmer le planning.", "label": "action"}, {"text": "Parfait, merci à vous.", "label": "information"}, {"text": "Bonjour, point sur l'avancement du bâtiment B.", "label": "information"}, {"text": "Le gros œuvre est terminé au troisième étage.", "label": "information"}, {"text": "On a eu deux jours de retard à cause de la pluie.", "label": "information"}, {"text": "La livraison du béton pour le quatrième étage est prévue mercredi.", "label": "information"}, {"text": "On maintient la date de fin du gros œuvre au 18 décembre.", "label": "decision"}, {"text": "Monsieur Amrani, renforcez l'équipe avec trois ouvriers cette semaine.", "label": "action"}, {"text": "Je contacte l'entreprise sous-traitante aujourd'hui.", "label": "action"}, {"text": "L'inspection de sécurité a relevé l'absence de garde-corps au troisième.", "label": "information"}, {"text": "Personne ne monte au troisième sans garde-corps, c'est non négociable.", "label": "decision"}, {"text": "Je fais installer les garde-corps demain.", "label": "action"}, {"text": "Je vais refaire une inspection jeudi après-midi.", "label": "action"}, {"text": "Le client viendra visiter le chantier en fin de mois.", "label": "information"}, {"text": "Il faut nettoyer le site avant la visite du client.", "label": "action"}, {"text": "Merci, on se revoit lundi.", "label": "information"}, {"text": "Bienvenue au sprint planning, on commence le sprint 8 aujourd'hui.", "label": "information"}, {"text": "Le sprint 7 s'est bien passé, on a livré neuf tickets sur dix.", "label": "information"}, {"text": "Le ticket manquant concerne la connexion avec Google.", "label": "information"}, {"text": "On reprend le ticket de connexion Google en priorité dans ce sprint.", "label": "decision"}, {"text": "Salma, tu prends le ticket de connexion Google.", "label": "action"}, {"text": "D'accord, je le termine pour mercredi.", "label": "action"}, {"text": "J'ai remarqué que les tests automatiques prennent vingt minutes.", "label": "information"}, {"text": "C'est beaucoup trop long pour chaque commit.", "label": "information"}, {"text": "On passe les tests lents en exécution nocturne.", "label": "decision"}, {"text": "Je configure l'exécution nocturne dans la CI cette semaine.", "label": "action"}, {"text": "Est-ce qu'on fait la démo au client à la fin du sprint ?", "label": "information"}, {"text": "Oui, la démo est fixée au vendredi 27 novembre.", "label": "decision"}, {"text": "Je prépare les slides de la démo.", "label": "action"}, {"text": "C'est bon pour moi, bon sprint à tous.", "label": "information"}, {"text": "Bonsoir, nous allons traiter l'éclairage public et le marché hebdomadaire.", "label": "information"}, {"text": "Trois lampadaires de la rue principale sont en panne depuis un mois.", "label": "information"}, {"text": "Les habitants ont peur de rentrer tard le soir.", "label": "information"}, {"text": "Le conseil adopte à l'unanimité la demande de réparation urgente.", "label": "decision"}, {"text": "Madame Idrissi, rédigez le courrier à la commune pour mardi.", "label": "action"}, {"text": "Je le rédige ce soir et je vous l'envoie.", "label": "action"}, {"text": "Pour le marché, les commerçants demandent plus de places.", "label": "information"}, {"text": "Le parking de l'école pourrait servir le dimanche.", "label": "information"}, {"text": "Nous retenons le parking de l'école pour agrandir le marché.", "label": "decision"}, {"text": "Je rencontre le directeur de l'école jeudi pour avoir son accord.", "label": "action"}, {"text": "Il faudra aussi prévoir des poubelles supplémentaires.", "label": "information"}, {"text": "Monsieur Lahlou, ajoutez ce point à votre rendez-vous avec le directeur.", "label": "action"}, {"text": "Merci à tous, la séance est levée.", "label": "information"}, {"text": "Salam, lyouma ghadi nhedro 3la la carte jdida w 3la les horaires.", "label": "information"}, {"text": "Les clients kaybghiw plats healthy ktar, b7al les salades.", "label": "information"}, {"text": "W chno 9alou 3la le tajine dyal l-7ut ?", "label": "information"}, {"text": "Bzaf dyal les clients 3jbhom, walakin ghali chwiya.", "label": "information"}, {"text": "Safi, ttafa9na nzidou 3 salades f la carte.", "label": "decision"}, {"text": "9rrna n9ssou le prix dyal le tajine l 70 dh.", "label": "decision"}, {"text": "Zineb, khassek tsifti la carte jdida l l-imprimerie 9bel jem3a.", "label": "action"}, {"text": "Wakha, ghadi nsiftha ghedda.", "label": "action"}, {"text": "Les weekends kayn bzaf dyal l-khdma, w 7na ghir joj f la cuisine.", "label": "information"}, {"text": "Ana ghadi nqelleb 3la cuisinier zayed had simana.", "label": "action"}, {"text": "W ila ma lqinach ?", "label": "information"}, {"text": "Nchoufou mn b3d.", "label": "information"}, {"text": "Hamid, dir lia lista dyal les produits li khasshom 9bel l-khmis.", "label": "action"}, {"text": "Merci l-kolchi.", "label": "information"}, {"text": "Good morning everyone, let's review the mobile app roadmap.", "label": "information"}, {"text": "The beta testers reported crashes on older Android phones.", "label": "information"}, {"text": "How many users are affected?", "label": "information"}, {"text": "Around eight percent of our active users.", "label": "information"}, {"text": "We decided to drop support for Android versions older than ten.", "label": "decision"}, {"text": "Ben, can you update the minimum version in the store listing by Friday?", "label": "action"}, {"text": "Sure, I'll do it tomorrow.", "label": "action"}, {"text": "Marketing wants the new onboarding before the holiday campaign.", "label": "information"}, {"text": "We agreed to ship the new onboarding on December 10.", "label": "decision"}, {"text": "I will prepare the release notes next week.", "label": "action"}, {"text": "Please send the campaign brief to the design team today.", "label": "action"}, {"text": "Should we also translate the app into Arabic?", "label": "information"}, {"text": "Maybe next quarter, let's discuss it later.", "label": "information"}, {"text": "Thanks everyone.", "label": "information"}, {"text": "Bonsoir à toutes et à tous.", "label": "information"}, {"text": "On a trois sujets ce soir: la collecte de fournitures scolaires, le local et les nouvelles adhésions.", "label": "information"}, {"text": "Concernant la collecte, l'année dernière on avait aidé 140 élèves à Sidi Moumen.", "label": "information"}, {"text": "Cette année les demandes sont plus nombreuses, les écoles nous ont signalé environ 200 élèves.", "label": "information"}, {"text": "Il faudrait alors deux points de collecte au lieu d'un, peut-être au lycée et à la mairie.", "label": "information"}, {"text": "Je propose qu'on garde les deux points de collecte.", "label": "information"}, {"text": "Vous êtes d'accord?", "label": "information"}, {"text": "Oui, d'accord.", "label": "information"}, {"text": "Très bien, la décision est prise: deux points de collecte, au lycée et à la mairie.", "label": "decision"}, {"text": "Hajar, tu peux contacter le directeur du lycée cette semaine?", "label": "action"}, {"text": "Oui, je l'appelle demain matin.", "label": "action"}, {"text": "Je me charge de demander l'autorisation à la mairie avant le 30 novembre.", "label": "action"}, {"text": "Deuxième sujet, le local.", "label": "information"}, {"text": "Le propriétaire veut augmenter le loyer de 15 %.", "label": "information"}, {"text": "C'est trop pour nous, notre budget est déjà serré.", "label": "information"}, {"text": "On pourrait demander une salle à la maison des jeunes, c'est gratuit pour les associations.", "label": "information"}, {"text": "On ne renouvelle pas le bail et on fait la demande à la maison des jeunes.", "label": "decision"}, {"text": "Je vais écrire au propriétaire pour l'informer d'ici la fin du mois.", "label": "action"}, {"text": "Et qui dépose le dossier à la maison des jeunes?", "label": "information"}, {"text": "Je peux le faire, j'ai les documents de l'association.", "label": "action"}, {"text": "Merci Hajar.", "label": "information"}, {"text": "Dernier point, les adhésions: on a reçu 12 nouvelles demandes.", "label": "information"}, {"text": "Il faut mettre à jour le registre des membres et encaisser les cotisations.", "label": "action"}, {"text": "Youssef, tu t'occupes du registre et des cotisations pour la prochaine réunion.", "label": "action"}, {"text": "Pas de problème.", "label": "information"}, {"text": "La prochaine réunion aura lieu le 5 décembre.", "label": "information"}, {"text": "Merci et bonne soirée.", "label": "information"}, {"text": "Salam l'équipe, lyouma ghadi nhedro 3la la livraison dyal l'application w 3la le client.", "label": "information"}, {"text": "Salam.", "label": "information"}, {"text": "Le client sift lina mail lbarh, bgha ybeddel la couleur dyal l'interface w yzid une page contact.", "label": "information"}, {"text": "Wach hadchi kayn f le contrat?", "label": "information"}, {"text": "La page contact machi f le contrat.", "label": "information"}, {"text": "Ok, ttafa9na: la couleur nbeddlouha, walakin la page contact ghadi nfactouriwha à part.", "label": "decision"}, {"text": "D'accord m3ak.", "label": "information"}, {"text": "Sara, khassek tsifti devis l le client 9bel jem3a.", "label": "action"}, {"text": "Ok, ghadi nsiftou ghedda.", "label": "action"}, {"text": "Ana ghadi nbeddel les couleurs d'ici lundi.", "label": "action"}, {"text": "Mzyan.", "label": "information"}, {"text": "La livraison, wach n9dro nsallmouha f 30 novembre?", "label": "information"}, {"text": "Ila ma kanch chi bug kbir, iyeh.", "label": "information"}, {"text": "9rrna: la livraison f 30 novembre.", "label": "decision"}, {"text": "Ana ghadi n3ayet l le client bach nconfirmi la date.", "label": "action"}, {"text": "Safi, merci.", "label": "information"}, {"text": "Bon, on commence?", "label": "information"}, {"text": "L'objectif aujourd'hui c'est de caler la campagne de lancement de l'application mobile et de faire le point sur le budget.", "label": "information"}, {"text": "Alors côté budget, on a dépensé 18 000 dirhams sur les 30 000 prévus pour le trimestre.", "label": "information"}, {"text": "La publicité Instagram a coûté plus cher que prévu, à peu près 7 000 dirhams au lieu de 5 000.", "label": "information"}, {"text": "Et ça a ramené combien d'inscriptions?", "label": "information"}, {"text": "Environ 1 200 inscriptions, donc un coût par inscription autour de 6 dirhams.", "label": "information"}, {"text": "C'est correct, mais TikTok marche mieux chez les 18-25 ans d'après nos tests.", "label": "information"}, {"text": "Donc on réduit Instagram et on met le reste du budget publicitaire sur TikTok pour le lancement.", "label": "decision"}, {"text": "Ça me va.", "label": "information"}, {"text": "Leila, tu prépares trois vidéos courtes pour TikTok d'ici le 25 novembre?", "label": "action"}, {"text": "Oui je m'en occupe, je te les montre avant de les publier.", "label": "action"}, {"text": "Parfait.", "label": "information"}, {"text": "Deuxième point, la date de lancement.", "label": "information"}, {"text": "Côté technique, la version 1.0 sera prête le 1er décembre si les tests de paiement passent.", "label": "information"}, {"text": "On a encore un bug sur le paiement par carte avec certaines banques.", "label": "information"}, {"text": "C'est bloquant?", "label": "information"}, {"text": "Oui, on ne peut pas lancer sans paiement fiable.", "label": "information"}, {"text": "Alors on fixe le lancement au 8 décembre, ça laisse une semaine de marge.", "label": "decision"}, {"text": "D'accord pour le 8.", "label": "information"}, {"text": "Mehdi, il faut corriger le bug de paiement avant le 1er décembre.", "label": "action"}, {"text": "Ok, je vais aussi faire tester l'application par cinq utilisateurs la semaine prochaine.", "label": "action"}, {"text": "Et pour la presse?", "label": "information"}, {"text": "On envoie un communiqué?", "label": "information"}, {"text": "Oui, Omar va rédiger le communiqué de presse pour le 4 décembre.", "label": "action"}, {"text": "Entendu.", "label": "information"}, {"text": "Moi je pense qu'on pourrait aussi faire un partenariat avec des influenceurs marocains.", "label": "information"}, {"text": "C'est une bonne idée mais on n'a pas le budget ce trimestre, on en reparle en janvier.", "label": "decision"}, {"text": "Merci à tous, je vous envoie le compte rendu.", "label": "action"}, {"text": "Bonjour à tous, merci d'être là.", "label": "information"}, {"text": "Alors aujourd'hui on a trois points: le choix du modèle de résumé, la maquette de l'interface et le planning de la démo.", "label": "information"}, {"text": "Bonjour.", "label": "information"}, {"text": "Oui, pour le modèle de résumé j'ai testé BARThez et mT5 sur une dizaine de transcriptions.", "label": "information"}, {"text": "BARThez donne de meilleurs résultats en français, le score ROUGE-L est autour de 0,38 alors que mT5 est à 0,31.", "label": "information"}, {"text": "Et le temps de traitement?", "label": "information"}, {"text": "Parce que, on doit rester sous une minute pour une heure de réunion.", "label": "information"}, {"text": "BARThez prend environ quarante secondes sur GPU pour une heure de réunion, donc ça passe.", "label": "information"}, {"text": "Ok, très bien.", "label": "information"}, {"text": "Donc on part sur BARThez pour le résumé, et on garde Mistral en option si le résultat n'est pas assez bon.", "label": "decision"}, {"text": "D'accord, ça me va.", "label": "information"}, {"text": "Yousra, tu peux préparer un petit tableau comparatif des deux modèles avant vendredi?", "label": "action"}, {"text": "Oui, je m'en occupe.", "label": "action"}, {"text": "Deuxième point, la maquette de l'interface.", "label": "information"}, {"text": "Rania, tu en es où?", "label": "information"}, {"text": "J'ai fait une première version sur Streamlit avec l'import de fichier et l'affichage du résumé.", "label": "information"}, {"text": "Il manque encore le tableau des actions et le bouton d'export en PDF.", "label": "information"}, {"text": "Est-ce qu'on peut ajouter une page pour l'historique des réunions?", "label": "information"}, {"text": "Les utilisateurs voudront retrouver les anciennes réunions.", "label": "information"}, {"text": "Oui c'est une bonne idée.", "label": "information"}, {"text": "Il est décidé d'ajouter une page historique avec une recherche par mot-clé.", "label": "decision"}, {"text": "Je vais ajouter le tableau des actions et l'export PDF d'ici mercredi prochain.", "label": "action"}, {"text": "Je me charge de la page historique, je peux la finir pour le 20 novembre.", "label": "action"}, {"text": "Parfait.", "label": "information"}, {"text": "Troisième point, le planning de la démo devant l'encadrant.", "label": "information"}, {"text": "L'encadrant est disponible le 2 décembre ou le 9 décembre.", "label": "information"}, {"text": "Le 2 décembre c'est un peu juste, on n'aura pas fini l'évaluation sur le corpus AMI.", "label": "information"}, {"text": "Bon, on valide le 9 décembre pour la démo, tout le monde est d'accord?", "label": "decision"}, {"text": "Oui, c'est validé pour moi.", "label": "information"}, {"text": "Salma, tu peux envoyer l'invitation à l'encadrant demain?", "label": "action"}, {"text": "Ok pas de souci.", "label": "information"}, {"text": "Et il faut que Yousra lance l'évaluation ROUGE sur le corpus AMI d'ici la fin du mois.", "label": "action"}, {"text": "Ça marche, je lance ça ce week-end.", "label": "action"}, {"text": "Super, merci à tous, on se revoit la semaine prochaine.", "label": "information"}, {"text": "On a décidé de garder PostgreSQL pour la base de données.", "label": "decision"}, {"text": "Il est convenu que la démo aura lieu le 9 décembre.", "label": "decision"}, {"text": "On part sur Streamlit pour le prototype.", "label": "decision"}, {"text": "C'est validé, on utilise BARThez pour le résumé.", "label": "decision"}, {"text": "Finalement on abandonne l'option React pour cette version.", "label": "decision"}, {"text": "Nous avons choisi le corpus AMI pour l'évaluation.", "label": "decision"}, {"text": "On valide le budget de 2000 euros pour les serveurs.", "label": "decision"}, {"text": "Tout le monde est d'accord pour décaler la réunion à jeudi.", "label": "decision"}, {"text": "La solution retenue est d'héberger le modèle en local.", "label": "decision"}, {"text": "On reporte la mise en production au mois de janvier.", "label": "decision"}, {"text": "Il a été décidé de ne pas stocker les fichiers audio.", "label": "decision"}, {"text": "On fixe la réunion hebdomadaire le lundi à dix heures.", "label": "decision"}, {"text": "C'est acté, la version 1 ne gère que le français.", "label": "decision"}, {"text": "On adopte le format PDF comme export par défaut.", "label": "decision"}, {"text": "Décision : on limite la démo à trois fonctionnalités.", "label": "decision"}, {"text": "On garde Mistral comme alternative au résumé abstractif.", "label": "decision"}, {"text": "Le comité a approuvé le recrutement d'un stagiaire.", "label": "decision"}, {"text": "On opte pour un entraînement sur GPU Colab.", "label": "decision"}, {"text": "Donc on retient la deuxième maquette.", "label": "decision"}, {"text": "We decided to postpone the release to next month.", "label": "decision"}, {"text": "Je vais préparer le tableau comparatif avant vendredi.", "label": "action"}, {"text": "Rania, tu peux envoyer le compte rendu aux participants ?", "label": "action"}, {"text": "Je m'occupe de la page historique d'ici mercredi.", "label": "action"}, {"text": "Salma se charge de réserver la salle pour la démo.", "label": "action"}, {"text": "Il faut que Yousra lance l'évaluation ROUGE d'ici la fin du mois.", "label": "action"}, {"text": "Je me charge de contacter l'encadrant demain.", "label": "action"}, {"text": "Karim va corriger les bugs de l'export Word.", "label": "action"}, {"text": "Vous pouvez relire le rapport avant le 15 novembre ?", "label": "action"}, {"text": "Merci de remplir le fichier d'annotation pour lundi.", "label": "action"}, {"text": "Je t'envoie la maquette ce soir.", "label": "action"}, {"text": "Tu pourrais vérifier les dates dans le planning ?", "label": "action"}, {"text": "N'oubliez pas de déposer vos slides sur le drive avant jeudi.", "label": "action"}, {"text": "Je finalise la documentation de l'API la semaine prochaine.", "label": "action"}, {"text": "Yousra doit annoter cinquante transcriptions supplémentaires.", "label": "action"}, {"text": "Je relance le fournisseur pour le devis.", "label": "action"}, {"text": "Pense à mettre à jour le README.", "label": "action"}, {"text": "Je fais la présentation pour la soutenance.", "label": "action"}, {"text": "Can you send me the slides by Friday?", "label": "action"}, {"text": "I'll handle the deployment tomorrow.", "label": "action"}, {"text": "Je prépare les données d'entraînement d'ici dimanche.", "label": "action"}, {"text": "Bonjour à tous, merci d'être présents.", "label": "information"}, {"text": "Le score ROUGE-L de BARThez est autour de 0,38.", "label": "information"}, {"text": "mT5 donne de moins bons résultats en français.", "label": "information"}, {"text": "La première version de l'interface affiche le résumé.", "label": "information"}, {"text": "Le corpus AMI contient une centaine d'heures de réunions.", "label": "information"}, {"text": "On a eu quelques problèmes avec la transcription des accents.", "label": "information"}, {"text": "Whisper prend environ quarante secondes pour une heure d'audio.", "label": "information"}, {"text": "L'encadrant est disponible le 2 ou le 9 décembre.", "label": "information"}, {"text": "Le serveur de test a été redémarré hier.", "label": "information"}, {"text": "Les utilisateurs se plaignent du temps de chargement.", "label": "information"}, {"text": "Est-ce qu'on a des retours sur la maquette ?", "label": "information"}, {"text": "Je pense que c'est une bonne idée.", "label": "information"}, {"text": "Le client a répondu à notre e-mail ce matin.", "label": "information"}, {"text": "Il y a trois points à l'ordre du jour.", "label": "information"}, {"text": "Le modèle spaCy ne détecte pas toujours les noms marocains.", "label": "information"}, {"text": "Les tests unitaires passent tous.", "label": "information"}, {"text": "La réunion précédente a duré une heure et demie.", "label": "information"}, {"text": "On en a déjà parlé la semaine dernière.", "label": "information"}, {"text": "Ça me paraît un peu juste comme délai.", "label": "information"}, {"text": "Le budget actuel couvre seulement les licences.", "label": "information"}, {"text": "Je ne suis pas sûr que ce soit la meilleure approche.", "label": "information"}, {"text": "Les résultats de la semaine sont encourageants.", "label": "information"}, {"text": "On n'aura pas fini l'évaluation pour le 2 décembre.", "label": "information"}, {"text": "Le fichier contient environ trois mille phrases.", "label": "information"}, {"text": "Merci à tous, bonne fin de journée.", "label": "information"}, {"text": "Après discussion, on retient le fournisseur B.", "label": "decision"}, {"text": "On arrête le développement de la version web.", "label": "decision"}, {"text": "Le comité valide le planning proposé.", "label": "decision"}, {"text": "Nous avons convenu de passer à deux réunions par mois.", "label": "decision"}, {"text": "On décide de recruter un développeur backend.", "label": "decision"}, {"text": "C'est réglé, on garde l'hébergement actuel.", "label": "decision"}, {"text": "On met en pause le projet mobile jusqu'en mars.", "label": "decision"}, {"text": "La direction a approuvé le nouveau budget.", "label": "decision"}, {"text": "On choisit la date du 12 janvier pour la formation.", "label": "decision"}, {"text": "Il est convenu que chaque équipe présente ses résultats vendredi.", "label": "decision"}, {"text": "On passe à Python 3.12 pour tout le projet.", "label": "decision"}, {"text": "L'assemblée adopte le règlement intérieur à l'unanimité.", "label": "decision"}, {"text": "Bon, on part sur la maquette numéro deux.", "label": "decision"}, {"text": "Alors on annule l'événement du mois prochain.", "label": "decision"}, {"text": "On maintient la réunion de jeudi malgré les absences.", "label": "decision"}, {"text": "Décision prise : les comptes rendus seront envoyés sous 24 heures.", "label": "decision"}, {"text": "On abandonne l'idée du partenariat pour cette année.", "label": "decision"}, {"text": "On fige le périmètre de la version 1.", "label": "decision"}, {"text": "Le jury a validé le sujet du projet.", "label": "decision"}, {"text": "On reporte la soutenance au 20 juin.", "label": "decision"}, {"text": "Donc on garde Whisper pour la transcription.", "label": "decision"}, {"text": "Tout le monde est d'accord pour utiliser Teams.", "label": "decision"}, {"text": "On opte pour un abonnement annuel.", "label": "decision"}, {"text": "Finalement on garde le même prestataire.", "label": "decision"}, {"text": "We agreed to use PostgreSQL in production.", "label": "decision"}, {"text": "It's decided, the launch is on Monday.", "label": "decision"}, {"text": "Tfahmna nkhdmo b Streamlit f la première version.", "label": "decision"}, {"text": "9rrna nbeddlo le fournisseur.", "label": "decision"}, {"text": "Ttafa9na 3la la date dyal la démo.", "label": "decision"}, {"text": "On en reparle à la prochaine réunion, ce n'est pas prioritaire.", "label": "decision"}, {"text": "Peux-tu mettre à jour le tableau de bord d'ici jeudi ?", "label": "action"}, {"text": "Je m'occupe de réserver la salle.", "label": "action"}, {"text": "Hamza va installer le serveur de test lundi.", "label": "action"}, {"text": "Je t'envoie les chiffres ce soir.", "label": "action"}, {"text": "Vous pouvez valider le document avant midi ?", "label": "action"}, {"text": "Il faut envoyer la facture au client avant la fin du mois.", "label": "action"}, {"text": "Je contacte le service juridique demain.", "label": "action"}, {"text": "Imane se charge de la communication sur LinkedIn.", "label": "action"}, {"text": "Je vais rédiger le cahier des charges cette semaine.", "label": "action"}, {"text": "Tu peux appeler le fournisseur pour le devis ?", "label": "action"}, {"text": "Merci de déposer vos rapports sur le drive avant vendredi.", "label": "action"}, {"text": "Je prépare la présentation pour le comité.", "label": "action"}, {"text": "N'oublie pas de mettre à jour le planning.", "label": "action"}, {"text": "Je relance l'équipe technique pour le bug de connexion.", "label": "action"}, {"text": "Zineb doit finaliser les maquettes pour le 3 décembre.", "label": "action"}, {"text": "Je corrige les fautes dans le rapport d'ici demain.", "label": "action"}, {"text": "Pourriez-vous m'envoyer vos disponibilités ?", "label": "action"}, {"text": "Je fais un point avec le client mardi.", "label": "action"}, {"text": "Il faut qu'on teste l'export Word avant la démo.", "label": "action"}, {"text": "Je me charge d'annoter les cent premières transcriptions.", "label": "action"}, {"text": "Rachid, tu t'occupes des achats de matériel ?", "label": "action"}, {"text": "Je vérifie les accès au serveur.", "label": "action"}, {"text": "On doit envoyer le dossier avant le 15 janvier.", "label": "action"}, {"text": "Je mets en place l'environnement Docker.", "label": "action"}, {"text": "Tu prépares le compte rendu de la séance ?", "label": "action"}, {"text": "Je vais tester BARThez sur le corpus AMI.", "label": "action"}, {"text": "Salma va organiser la réunion avec l'encadrant.", "label": "action"}, {"text": "Pense à sauvegarder la base de données.", "label": "action"}, {"text": "Je finalise le chapitre trois du rapport ce week-end.", "label": "action"}, {"text": "Ghadi nsifet lik le fichier daba.", "label": "action"}, {"text": "Khassek tkemmel la maquette 9bel tnin.", "label": "action"}, {"text": "Ana ghadi nhder m3a le client ghedda.", "label": "action"}, {"text": "Khassna nsaliw les tests 9bel la démo.", "label": "action"}, {"text": "I will update the documentation tomorrow.", "label": "action"}, {"text": "Could you review my pull request?", "label": "action"}, {"text": "Please send the invoice by Friday.", "label": "action"}, {"text": "Je peux m'en charger.", "label": "action"}, {"text": "Oui, je m'en occupe dès demain.", "label": "action"}, {"text": "Je regarde ça et je te redis.", "label": "action"}, {"text": "Je reprends le module d'export.", "label": "action"}, {"text": "Le serveur a planté deux fois cette semaine.", "label": "information"}, {"text": "On a reçu une réponse positive du client.", "label": "information"}, {"text": "Le taux de conversion est passé à 4 %.", "label": "information"}, {"text": "Je ne suis pas disponible jeudi.", "label": "information"}, {"text": "Les tests de charge ont donné de bons résultats.", "label": "information"}, {"text": "Il y a eu un retard de livraison du matériel.", "label": "information"}, {"text": "On en a parlé avec l'encadrant la semaine dernière.", "label": "information"}, {"text": "Le modèle met environ dix secondes par fichier.", "label": "information"}, {"text": "Qu'est-ce que vous en pensez ?", "label": "information"}, {"text": "Je trouve que l'interface est un peu chargée.", "label": "information"}, {"text": "On pourrait peut-être utiliser une autre couleur.", "label": "information"}, {"text": "Le rapport fait actuellement trente pages.", "label": "information"}, {"text": "Est-ce que quelqu'un a des questions ?", "label": "information"}, {"text": "La dernière version corrige le problème d'accents.", "label": "information"}, {"text": "Je pense que c'est faisable dans les délais.", "label": "information"}, {"text": "Le client n'a pas encore répondu.", "label": "information"}, {"text": "Ça dépend du budget disponible.", "label": "information"}, {"text": "On a eu 45 participants à l'atelier.", "label": "information"}, {"text": "C'est intéressant comme approche.", "label": "information"}, {"text": "Le fichier audio est de mauvaise qualité.", "label": "information"}, {"text": "Il pleuvait, donc peu de monde est venu.", "label": "information"}, {"text": "Le précédent prestataire était plus cher.", "label": "information"}, {"text": "On n'a pas encore les résultats de l'évaluation.", "label": "information"}, {"text": "Bienvenue à Imane qui rejoint l'équipe.", "label": "information"}, {"text": "On manque de données annotées en darija.", "label": "information"}, {"text": "Je ne sais pas si c'est la bonne solution.", "label": "information"}, {"text": "Il faudrait peut-être y réfléchir davantage.", "label": "information"}, {"text": "Oui, je suis d'accord avec toi.", "label": "information"}, {"text": "Merci pour la présentation.", "label": "information"}, {"text": "Le dernier sprint s'est bien passé.", "label": "information"}, {"text": "La salle est réservée jusqu'à 18 heures.", "label": "information"}, {"text": "Wach kolchi wajed ?", "label": "information"}, {"text": "Lbarh tsift lina le contrat.", "label": "information"}, {"text": "Hadchi mzyan bzaf.", "label": "information"}, {"text": "Le client ma jawbch mazal.", "label": "information"}, {"text": "The meeting is recorded.", "label": "information"}, {"text": "We had some issues with the API last week.", "label": "information"}, {"text": "I think it's a good idea.", "label": "information"}, {"text": "Ok, merci.", "label": "information"}, {"text": "On passe au point suivant.", "label": "information"}, {"text": "Je propose qu'on vote à la prochaine réunion.", "label": "information"}, {"text": "Est-ce qu'on change de fournisseur ?", "label": "information"}, {"text": "Les retours des utilisateurs sont plutôt positifs.", "label": "information"}, {"text": "Il reste trois semaines avant la soutenance.", "label": "information"}, {"text": "La version actuelle ne supporte pas l'audio.", "label": "information"}], "test": [{"text": "Bonjour, nous devons organiser la clôture des comptes de fin d'année.", "label": "information"}, {"text": "Nous avons quarante clients à clôturer avant le 31 décembre.", "label": "information"}, {"text": "L'année dernière, nous avions terminé avec une semaine de retard.", "label": "information"}, {"text": "Le principal problème, c'était les factures reçues trop tard.", "label": "information"}, {"text": "À partir de cette année, la date limite de dépôt des factures est fixée au 5 décembre.", "label": "decision"}, {"text": "Leila, informez tous les clients de cette nouvelle date d'ici mercredi.", "label": "action"}, {"text": "Très bien, j'envoie un e-mail groupé dès demain.", "label": "action"}, {"text": "Il faudrait peut-être aussi un logiciel de relance automatique.", "label": "information"}, {"text": "C'est intéressant, mais le budget est déjà engagé pour cette année.", "label": "information"}, {"text": "Le logiciel de relance est reporté à l'année prochaine.", "label": "decision"}, {"text": "Monsieur Ouazzani, préparez la répartition des dossiers par collaborateur pour vendredi.", "label": "action"}, {"text": "Je m'en charge.", "label": "action"}, {"text": "Est-ce qu'on prévoit des heures supplémentaires en décembre ?", "label": "information"}, {"text": "Oui, les heures supplémentaires sont autorisées jusqu'à la fin de la clôture.", "label": "decision"}, {"text": "Merci à tous.", "label": "information"}, {"text": "Bonjour, il faut planifier les soutenances des projets de fin d'année.", "label": "information"}, {"text": "Nous avons vingt-quatre binômes cette année.", "label": "information"}, {"text": "Chaque soutenance dure trente minutes avec les questions.", "label": "information"}, {"text": "Sur deux jours, ça fait douze soutenances par jour, c'est beaucoup.", "label": "information"}, {"text": "Nous organisons les soutenances sur trois jours, du 14 au 16 décembre.", "label": "decision"}, {"text": "Il faudra au moins trois salles équipées de vidéoprojecteurs.", "label": "information"}, {"text": "Docteur Naciri, pourriez-vous réserver les salles auprès de la scolarité ?", "label": "action"}, {"text": "Oui, je passe à la scolarité jeudi.", "label": "action"}, {"text": "Les étudiants demandent aussi s'ils peuvent rendre le rapport en retard.", "label": "information"}, {"text": "Non, aucun retard ne sera accepté pour les rapports.", "label": "decision"}, {"text": "Je publie le planning des soutenances sur la plateforme avant le 4 décembre.", "label": "action"}, {"text": "Et pour les jurys, qui fait la répartition ?", "label": "information"}, {"text": "Je m'occupe de la composition des jurys.", "label": "action"}, {"text": "Merci pour cette réunion efficace.", "label": "information"}, {"text": "Salut l'équipe, on parle des livraisons pour la période des fêtes.", "label": "information"}, {"text": "Les commandes ont augmenté de quarante pour cent depuis le Black Friday.", "label": "information"}, {"text": "Notre transporteur actuel n'arrive plus à suivre.", "label": "information"}, {"text": "J'ai reçu deux offres, une de Fast Delivery et une de MaroColis.", "label": "information"}, {"text": "MaroColis est un peu plus cher mais livre en vingt-quatre heures.", "label": "information"}, {"text": "Ok, on signe avec MaroColis pour décembre.", "label": "decision"}, {"text": "Amal, envoie-leur le contrat signé demain.", "label": "action"}, {"text": "Ça marche, je le fais demain matin.", "label": "action"}, {"text": "L'entrepôt est presque plein, on stocke des cartons dans le couloir.", "label": "information"}, {"text": "On loue un local supplémentaire pour deux mois.", "label": "decision"}, {"text": "Je cherche un local près de l'entrepôt d'ici vendredi.", "label": "action"}, {"text": "Il faudra aussi des préparateurs de commandes en plus.", "label": "information"}, {"text": "Younes, publie l'offre pour deux préparateurs ce week-end.", "label": "action"}, {"text": "Parfait, on fait le point lundi prochain.", "label": "information"}, {"text": "Salam l'équipe, khassna nwjjdou la campagne dyal Ramadan.", "label": "information"}, {"text": "3ndna budget dyal 20 000 dh l had la campagne.", "label": "information"}, {"text": "L3am li fat khdmna b les influenceurs, w ma jabouch bzaf d les ventes.", "label": "information"}, {"text": "Les pubs f Instagram w TikTok kanu a7ssan.", "label": "information"}, {"text": "Safi, 9rrna nkhdmo ghir b les pubs had l3am, bla influenceurs.", "label": "decision"}, {"text": "Sanaa, khassek twjjdi le planning dyal les pubs 9bel jem3a.", "label": "action"}, {"text": "Wakha, ghadi nwjjdou had simana.", "label": "action"}, {"text": "W les visuels, chkoun ghadi ydirhom ?", "label": "information"}, {"text": "Ana ghadi nsawb les visuels w nsiftom lik tnin.", "label": "action"}, {"text": "Ttafa9na tani nbdaw la campagne 15 jour 9bel Ramadan.", "label": "decision"}, {"text": "Wach n9dro nzidou chi promo f le site ?", "label": "information"}, {"text": "Nchoufou had l-mas2ala f la réunion jaya.", "label": "information"}, {"text": "Yallah, bon courage.", "label": "information"}, {"text": "Bonsoir à tous, l'hiver arrive et les familles du douar ont besoin de couvertures.", "label": "information"}, {"text": "L'année dernière, on avait distribué trois cents couvertures.", "label": "information"}, {"text": "Cette année, on a recensé cent vingt familles.", "label": "information"}, {"text": "Notre objectif est fixé à quatre cents couvertures.", "label": "decision"}, {"text": "Il nous faut environ vingt mille dirhams.", "label": "information"}, {"text": "Said, lance la cagnotte en ligne dès cette semaine.", "label": "action"}, {"text": "D'accord, je la mets en ligne mercredi.", "label": "action"}, {"text": "Le supermarché du centre accepte d'accueillir un point de collecte.", "label": "information"}, {"text": "On accepte la proposition du supermarché.", "label": "decision"}, {"text": "Je m'occupe de l'affiche pour le point de collecte.", "label": "action"}, {"text": "Fatine, il faudrait aussi trouver un camion pour la distribution avant la fin du mois.", "label": "action"}, {"text": "La distribution pourrait se faire le dernier week-end de décembre.", "label": "information"}, {"text": "C'est validé, la distribution aura lieu le 26 décembre.", "label": "decision"}, {"text": "Merci à tous pour votre engagement.", "label": "information"}]}
EPOCHS = 8
print(len(DATA['train']), 'train /', len(DATA['test']), 'test')

## Entraînement + évaluation sur le test

In [ ]:
# ==== MeetMind — fine-tuning CamemBERT (code exécuté dans Colab) ====
import json, inspect, os, numpy as np, torch
from datasets import Dataset
from sklearn.metrics import classification_report, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

MODEL_NAME = os.environ.get("MEETMIND_BASE_MODEL", "almanach/camembert-base")
OUT = "camembert-meetmind"
LABELS = ["information", "decision", "action"]; L2I = {l: i for i, l in enumerate(LABELS)}
set_seed(42)

train_rows, test_rows = DATA["train"], DATA["test"]
print(f"Entraînement : {len(train_rows)} phrases · Test : {len(test_rows)} phrases")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
def encode(b):
    e = tok(b["text"], truncation=True, max_length=128); e["labels"] = [L2I[l] for l in b["label"]]; return e
ds_tr = Dataset.from_list(train_rows).map(encode, batched=True, remove_columns=["text", "label"])
ds_te = Dataset.from_list(test_rows).map(encode, batched=True, remove_columns=["text", "label"])

counts = np.bincount([L2I[r["label"]] for r in train_rows], minlength=3)
weights = torch.tensor(counts.sum() / (3 * np.maximum(counts, 1)), dtype=torch.float)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=L2I)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

steps = max(1, len(train_rows) // 16) * EPOCHS
wanted = dict(output_dir="ckpt", num_train_epochs=EPOCHS, per_device_train_batch_size=16, per_device_eval_batch_size=32,
              learning_rate=3e-5, weight_decay=0.01, warmup_steps=int(0.1 * steps), eval_strategy="no",
              evaluation_strategy="no", save_strategy="no", logging_steps=10, seed=42, report_to=[])
ok = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" in ok: wanted.pop("evaluation_strategy")
args = TrainingArguments(**{k: v for k, v in wanted.items() if k in ok})
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_tr, data_collator=DataCollatorWithPadding(tok))
trainer.train()

pred = trainer.predict(ds_te).predictions.argmax(-1)
gold = [L2I[r["label"]] for r in test_rows]
print("\n===== CamemBERT sur les 70 phrases de TEST =====")
print(classification_report(gold, pred, target_names=LABELS, digits=3, zero_division=0))
p, r, f, _ = precision_recall_fscore_support(gold, pred, labels=[1, 2], zero_division=0)
RESULT = {"f1_decision": round(float(f[0]), 3), "f1_action": round(float(f[1]), 3)}
print("Comparaison (même test) :  Règles  F1 décision 0,59 · action 0,50")
print("                           TF-IDF  F1 décision 0,53 · action 0,90")
print(f"                           CamemBERT F1 décision {RESULT['f1_decision']} · action {RESULT['f1_action']}")
trainer.save_model(OUT); tok.save_pretrained(OUT)
json.dump(RESULT, open(f"{OUT}/resultats_test.json", "w"))
print(f"\nModèle sauvegardé dans {OUT}/")


## Vérification : le modèle se charge comme dans l'application

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model="camembert-meetmind")
for t in ["On a décidé de passer à PostgreSQL.", "Imane, tu peux envoyer le devis avant jeudi ?",
          "Le serveur a redémarré deux fois hier.", "Ttafa9na nbeddlo la date dyal la démo."]:
    p = clf(t)[0]; print(f"{p['label']:<12} {p['score']:.2f}  {t}")

## Télécharger le modèle

In [ ]:
!rm -f camembert-meetmind/training_args.bin && zip -qr camembert-meetmind.zip camembert-meetmind && ls -lh camembert-meetmind.zip
from google.colab import files
files.download("camembert-meetmind.zip")